In [1]:
import time
import pandas as pd

import sys, os

added_path = os.path.abspath("..")
sys.path.append(added_path)

# utils 모듈 임포트
from utils.naver_searchad_relkeyword import fetch_relkwdstat
from utils.naver_shoppinginsite_search import fetch_category_keyword_data

if added_path in sys.path:
    sys.path.remove(added_path)

INPUT_PATH = "../data/processed_keyword.csv"
OUTPUT_PATH = "../data/train_dataset.csv"
WEEKLY_OUTPUT_PATH = "../data/train_dataset_weekly.csv"   # ✅ 주간 단위 결과 저장 경로 추가

# ✅ 데이터 로드 및 기본 처리
df = pd.read_csv(INPUT_PATH)
df["collect_day"] = pd.to_datetime(df["collect_day"])
df["weekday"] = df["collect_day"].dt.weekday

# ✅ 원본 컬럼 정제
for col in ["일간검색수", "월평균클릭수", "월간검색수"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "")
            .astype(float)
        )

In [2]:
results = []

for good_id, group in df.groupby("good_id"):
    print(f"🔍 good_id: {good_id} → API 데이터 수집 중...")

    keyword = group["검색키워드"].iloc[0]

    # ✅ 1. RelKwdStat API 호출
    rel_data = fetch_relkwdstat([keyword])
    if rel_data:
        recent_4weeks_click_avg = rel_data[0].get("최근4주클릭수평균", None)  # 값 없으면 NaN 처리
    else:
        recent_4weeks_click_avg = None

    # ✅ 2. ShoppingInsight API 호출
    start_date = group["collect_day"].min().strftime("%Y-%m-%d")
    end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    try:
        search_df = fetch_category_keyword_data(
            start_date=start_date,
            end_date=end_date,
            category="50000006",
            keyword=keyword
        )
    except Exception as e:
        print(f"⚠️ API 호출 실패: {e}")
        search_df = pd.DataFrame(columns=["날짜", "클릭량"])

    print(f"✅ good_id: {good_id} → 병합 중...")

    if not search_df.empty:
        search_df["날짜"] = pd.to_datetime(search_df["날짜"])
    else:
        search_df = pd.DataFrame({"날짜": group["collect_day"], "클릭량": [None]*len(group)})

    merged = group.merge(search_df, left_on="collect_day", right_on="날짜", how="left")

    # ✅ 3. 지난달 클릭량 합계 계산
    if not search_df.empty and "클릭량" in search_df.columns and "날짜" in search_df.columns:
        today = pd.Timestamp.today()
        start_date = today - pd.DateOffset(months=1)
        end_date = today

        prev_month_df = search_df[
            (search_df["날짜"] >= start_date) & (search_df["날짜"] < end_date)
        ]

        prev_month_sum = prev_month_df["클릭량"].sum()
    else:
        prev_month_sum = None

    merged["지난달클릭량합계"] = prev_month_sum
    merged["최근4주클릭수평균"] = recent_4weeks_click_avg

    if recent_4weeks_click_avg is not None:
        recent_4weeks_click_total = recent_4weeks_click_avg * 30
    else:
        recent_4weeks_click_total = None

    merged["최근4주클릭수총합(30일기준)"] = recent_4weeks_click_total

    merged["최근4주클릭수_비율"] = merged.apply(
        lambda row: (recent_4weeks_click_total / prev_month_sum)
        if (recent_4weeks_click_total is not None and prev_month_sum not in [None, 0])
        else None,
        axis=1
    )

    # ✅ 추정일간클릭수 계산
    merged["추정일간클릭수"] = merged["최근4주클릭수_비율"] * merged["클릭량"]

    results.append(merged)
    time.sleep(0.3)

final_df = pd.concat(results, ignore_index=True)
print("병합 완료 ✅")

🔍 good_id: 9517327 → API 데이터 수집 중...
✅ good_id: 9517327 → 병합 중...
🔍 good_id: 11268388 → API 데이터 수집 중...
✅ good_id: 11268388 → 병합 중...
🔍 good_id: 11332531 → API 데이터 수집 중...
✅ good_id: 11332531 → 병합 중...
🔍 good_id: 11508428 → API 데이터 수집 중...
✅ good_id: 11508428 → 병합 중...
🔍 good_id: 17550423 → API 데이터 수집 중...
✅ good_id: 17550423 → 병합 중...
🔍 good_id: 21732507 → API 데이터 수집 중...
✅ good_id: 21732507 → 병합 중...
🔍 good_id: 32575528 → API 데이터 수집 중...
✅ good_id: 32575528 → 병합 중...
🔍 good_id: 81485426 → API 데이터 수집 중...
✅ good_id: 81485426 → 병합 중...
🔍 good_id: 95144100 → API 데이터 수집 중...
✅ good_id: 95144100 → 병합 중...
🔍 good_id: 101497091 → API 데이터 수집 중...
✅ good_id: 101497091 → 병합 중...
🔍 good_id: 103686207 → API 데이터 수집 중...
✅ good_id: 103686207 → 병합 중...
🔍 good_id: 104555112 → API 데이터 수집 중...
✅ good_id: 104555112 → 병합 중...
🔍 good_id: 105383486 → API 데이터 수집 중...
✅ good_id: 105383486 → 병합 중...
🔍 good_id: 105387395 → API 데이터 수집 중...
✅ good_id: 105387395 → 병합 중...
🔍 good_id: 114577278 → API 데이터 수집 중...
✅

In [3]:
pd.set_option("display.max_rows", None)   # 모든 행 출력
pd.set_option("display.max_columns", None)  # 모든 열 출력
grouped = final_df.groupby(["good_id","weekday"])["클릭량"].mean().unstack()
grouped

weekday,0,1,2,3,4,5,6
good_id,,,,,,,
9517327,35.515948,33.223962,29.273267,27.669980,23.489957,22.602055,30.047697
11268388,22.173913,17.333333,21.090909,20.857143,20.555556,21.300000,18.341463
11332531,31.032253,32.387091,27.715597,27.999995,27.254769,21.908597,29.427249
11508428,34.755444,37.607199,35.509637,31.804403,30.853989,27.963844,32.968069
17550423,27.005340,18.716569,24.242417,22.443174,23.896096,22.272720,18.434335
21732507,35.462473,33.223962,29.790457,27.669980,23.716109,22.602055,30.047697
32575528,29.575159,20.735290,20.588230,19.327726,14.285710,20.142598,16.563463
81485426,24.297867,25.705596,22.778467,21.744675,18.882972,17.952121,22.145022
95144100,27.260977,28.282824,26.641409,25.736957,26.422760,22.222218,21.963820


In [4]:
cleaned_final_df = final_df[
    ~final_df.isna().any(axis=1) & 
    (final_df["지난달클릭량합계"] != 0) & 
    (final_df["최근4주클릭수평균"] != 0)
]

print("최종 정제된 데이터 크기:", cleaned_final_df.shape)
print(cleaned_final_df.head())

최종 정제된 데이터 크기: (25186, 26)
  collect_day  good_id  pum_id pum_name                  good_name  \
0  2025-02-01  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
1  2025-02-02  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
2  2025-02-03  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
3  2025-02-04  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
4  2025-02-05  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   

   sales_price  discount_price  benefit 검색키워드  1개당가격  카탈로그매칭가격      카탈로그 ID  \
0        36900           36900        0    햇반   1025     24600  55379805801   
1        36900           36900        0    햇반   1025     24600  55379805801   
2        36900           36900        0    햇반   1025     24600  55379805801   
3        36900           36900        0    햇반   1025     24600  55379805801   
4        36900           36900        0    햇반   1025     24600  55379805801   

    일간검색수  일평균클릭수 일평균클릭률    월간검색수  월평균클릭수 월평균클릭률  weekday         날짜  \
0  27

In [5]:
cleaned_final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"✅ 최종 결과 저장 완료 → {OUTPUT_PATH}")

✅ 최종 결과 저장 완료 → ../data/train_dataset.csv


In [ ]:
# ✅ 주간 단위 그룹화 결과 저장
final_df["week"] = final_df["collect_day"].dt.isocalendar().week
final_df["year"] = final_df["collect_day"].dt.isocalendar().year

agg_dict = {
    "collect_day": "min",   # 주차 시작일
    "pum_id": "first",
    "pum_name": "first",
    "good_name": "first",
    "sales_price": "first",
    "discount_price": "first",
    "benefit": "first",
    "검색키워드": "first",
    "1개당가격": "first",
    "카탈로그매칭가격": "first",
    "카탈로그 ID": "first",

    # 숫자형 지표 → 평균
    "일간검색수": "mean",
    "일평균클릭수": "mean",
    "월간검색수": "mean",
    "월평균클릭수": "mean",
    "클릭량": "mean",
    "추정일간클릭수": "mean",

    # 문자열/퍼센트 컬럼 → 대표값 유지
    "일평균클릭률": "first",
    "월평균클릭률": "first",
    # 고정값/참조용 → 대표값 유지
    "지난달클릭량합계": "first",
    "최근4주클릭수평균": "first",
    "최근4주클릭수총합(30일기준)": "first",
    "최근4주클릭수_비율": "first",
}

weekly_df = (
    final_df.groupby(["good_id", "year", "week"])
    .agg(agg_dict)
    .reset_index()
)

weekly_df.to_csv(WEEKLY_OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"✅ 주간 단위 결과 저장 완료 → {WEEKLY_OUTPUT_PATH}")